<a href="https://colab.research.google.com/github/thatcherty/ML-Algo-Selection/blob/main/ML_Algo_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Algo Selection

In [1]:
# Common
import pathlib
import json
import importlib
import warnings
import pandas as pd
import numpy as np
from typing import Optional


from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, accuracy_score
from sklearn.exceptions import ConvergenceWarning

import tensorflow as tf

2026-03-31 18:05:31.705381: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 18:05:31.715248: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775009131.727426  118647 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775009131.730793  118647 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775009131.739993  118647 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# The code below was modified for development and to help with logging and error handling.
# It should not be considered finished or complete. As with the main branch, it does not run 
# correctly for all of the ~100 datasets and use cases.
# The neural networks are in general less robust and require more configuration than some of the other ML methods.

import csce_ml_project_scraper
importlib.reload(csce_ml_project_scraper)

import csce_ml_project_dataset_manager
importlib.reload(csce_ml_project_dataset_manager)

import csce_ml_project_models
importlib.reload(csce_ml_project_models)



def extract_features(dataset):
    # Get dataset features
    name = dataset.metadata['name']
    feature_count = dataset.metadata['num_features']
    instance_count = dataset.metadata['num_instances']
    missing_values = dataset.metadata['has_missing_values']
    
    new_row = {'Dataset': name, 'Feature Count': feature_count, 'Instances': instance_count, 'Missing Values': missing_values}
    
    return new_row



# preprocessing
def build_preprocessor(X: pd.DataFrame):
    # Detect column types
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    # Numeric preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    # Combine both
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor



# simulate train and test
def run_simulation(X, y, model, name, dataset, log_all = 0, log_conf = 0) -> float:
    # preprocessor will determine what preprocessing to perform on what columns
    preprocessor = build_preprocessor(X)

    # preprocess data and select
    pipe = Pipeline([('preprocessor', preprocessor), (name, model)])

    # split data into 5 sections
    # select 4 of 5 for train and last for test
    # do this selection 5 times
    min_class_size = y.value_counts().min()

    if min_class_size < 2:
        print("Class size < 2")

    n_splits = min(5, min_class_size)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    # fits model on each of the 5 train/test splits
    # reports all accuracy and f1 for the 5 splits
    results = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=["f1_macro"],
        return_train_score=False,
    )

    # extract results
    f1_scores = results["test_f1_macro"]

    # get mean and std f1 among 5 splits
    # used to determine confidence score
    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)

    # determine confidence score
    # high f1 score and low std is preferred
    confidence_score = mean_f1 - std_f1

    # NOTE: dataset is not defined in this scope, which can cause problems.

    # optional logging
    if log_all:
        print(f"{name} {dataset.metadata["uci_id"]} mean F1: {mean_f1:.4f}")
        print(f"{name} {dataset.metadata["uci_id"]} std F1: {std_f1:.4f}")

    if log_all or log_conf:
        print(f"{name} {dataset.metadata["uci_id"]} confidence score (mean F1 - std F1): {confidence_score:.4f}")

    return confidence_score



# determine best model
def get_best_model(model_scores:dict):
    if len(model_scores.keys())<=0:
        print(f"There are no model scores.")
        return None, None
    best_model = max(model_scores, key=model_scores.get)    
    ties = [model for model, score in model_scores.items() if score == model_scores[best_model]]    
    # choose simpler model of tied models
    if len(ties) > 1:
        preference = ["Logistic", "Decision Tree", "Neural Network"]
        for model in preference:
            if model in ties:
                best_model = model
                break
    else:
      best_model = ties[0]  
    return best_model, model_scores[best_model]



def get_id_list(method:str='existing_list'):
    id_list = []
    if method=='scraper':
        id_list = csce_ml_project_dataset_scraper.run()
    elif method=='existing_list':
        id_list = [53, 45, 186, 17, 222, 2, 320, 352, 109, 19, 697, 350, 
                                144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 
                                602, 27, 20, 336, 242, 601, 292, 174, 327, 80, 545, 42, 
                                267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 
                                529, 579, 936, 46, 942, 848, 563, 890, 52, 878, 383, 225, 
                                572, 850, 887, 857, 445, 158, 151, 101, 571, 145, 938, 915, 
                                484, 880, 864, 365, 110, 547, 759, 244, 193, 732, 763, 33, 
                                105, 76, 143, 451, 176, 426, 603, 198, 12, 16, 264, 379, 
                                471, 760, 39, 212, 461, 172, 591, 467, 90, 536, 50, 503, 
                                47, 419, 161, 117, 81, 357, 485, 911, 312, 58, 342, 537, 
                                827, 30, 329, 582, 799, 247, 40, 43, 26, 565, 380, 149, 
                                146, 13, 38, 270, 372, 148, 913, 367, 229, 257, 277, 713, 
                                300, 54, 28, 83, 728, 722, 22, 567, 184, 78, 107, 95, 63, 
                                755, 3, 70, 44, 75, 69, 91, 23, 74, 32, 96, 82, 18, 88, 8, 147]
    else:
        id_list = [53, 45, 186, 17, 222, 2, 320, 352, 109, 19, 697, 350, 
                                144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 
                                602, 27, 20, 336, 242, 601, 292, 174, 327, 80, 545, 42, 
                                267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 
                                529, 579, 936, 46, 942, 848, 563, 890, 52, 878, 383, 225, 
                                572, 850, 887, 857, 445, 158, 151, 101, 571, 145, 938, 915, 
                                484, 880, 864, 365, 110, 547, 759, 244, 193, 732, 763, 33, 
                                105, 76, 143, 451, 176, 426, 603, 198, 12, 16, 264, 379, 
                                471, 760, 39, 212, 461, 172, 591, 467, 90, 536, 50, 503, 
                                47, 419, 161, 117, 81, 357, 485, 911, 312, 58, 342, 537, 
                                827, 30, 329, 582, 799, 247, 40, 43, 26, 565, 380, 149, 
                                146, 13, 38, 270, 372, 148, 913, 367, 229, 257, 277, 713, 
                                300, 54, 28, 83, 728, 722, 22, 567, 184, 78, 107, 95, 63, 
                                755, 3, 70, 44, 75, 69, 91, 23, 74, 32, 96, 82, 18, 88, 8, 147]
    return id_list



class RunConfig:


    def __init__(
            self, 
            model_list:Optional[list[str]]=None,
            model_dict:Optional[dict]=None,
            max_datasets:Optional[int]=None, 
            max_rows:Optional[int]=None,
            method:Optional[str|int]|None=None,
            id_list:Optional[list[str]]=None,
            first_id:Optional[int]=None,
            filterwarnings:Optional[bool]|None=False,
            ):
        self.model_dict = {}
        self.default_settings_dict = {
            'log_conf': True,
            'log_all' : False,
        }

        # Model list 
        # This is not necessarily the best way to do this, but it is a compromise with the existing program.
        default_model_list = [
            'test_log' ,
            'test_nn'  ,
            'test_nn2' ,
            'test_tree',
        ]
        if model_list is None:
            model_list = default_model_list
        if not isinstance(model_list, list) and (len(model_list)>0) and (all([isinstance(model, str) for model in model_list])):
            model_list = default_model_list
        self.model_list = model_list
        if model_dict is None:
            for model in model_list:
                self.model_dict[model] = self.default_settings_dict.copy()
            self.max_datasets=max_datasets
        elif isinstance(model_dict, dict):
            self.model_dict = model_dict

        # Max rows to use for training, None -> no limit.
        if isinstance(max_rows, int):
            self.max_rows=max_rows
        else:
            self.max_rows=None

        # Method for getting an id list
        if isinstance(method, (str|int)):
            self.method = method
        
        # Id list if one is provided, otherwise obtain one.
        if id_list is None:
            id_list = get_id_list(self.method)

        # First id.
        if isinstance(first_id, (int|str|None)):
            self.first_id = first_id
        else:
            self.first_id = None

        # Whether or not to filter some warnings.
        if isinstance(filterwarnings, (bool,int)):
            self.filterwarnings = filterwarnings
        else:
            self.filterwarnings = False



def get_datasets(ids, use_dataset_loader, max_datasets=None):

    if isinstance(max_datasets, int):
        max_dataset_index = min([
            max_datasets,
            len(ids),
        ])
    else:
        max_dataset_index = len(ids)

    if not use_dataset_loader:
        print(f"Using remote datasets.")
        print(len(ids))
        datasets = []
        for dataset_id in ids[:max_dataset_index]:
            datasets.append(fetch_ucirepo(id=dataset_id))
        return datasets
    else:
        print(f"Using local datasets.")
        ds = csce_ml_project_dataset_manager.DataSetManager()
        ds.load(ids[:max_dataset_index])
        datasets = ds.get_dataset_list()
        return datasets



def run_switch(method='existing_list', max_datasets=None, use_dataset_loader=True, filterwarnings=True):

    ids = get_id_list(method=method)

    datasets = get_datasets(ids=ids, use_dataset_loader=use_dataset_loader,max_datasets=max_datasets)

    # exclude convergence warnings for clearer logging
    if filterwarnings:
        warnings.filterwarnings("ignore", category=ConvergenceWarning)

    test_log  = 0
    test_nn   = 0
    test_nn2  = 0
    test_tree = 0

    # Create final dataset obj
    algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": [], "Expected Confidence": []})
    algo_selection["Recommended Algorithm"] = algo_selection["Recommended Algorithm"].astype("string")

    # test_log = 1
    # test_nn = 1
    test_nn2  = 1
    test_tree = 1

    # for dataset in datasets[:max_dataset_index]:
    for dataset in datasets:

        model_scores = {}

        # Get the targets for the current dataset
        current_dataset_targets_df = dataset.data.targets

        # This should be done by exclusion previously, but keep just in case
        # Skip if no target variables or if the target DataFrame is empty
        if current_dataset_targets_df is None or current_dataset_targets_df.empty:
            print(f"Skipping dataset {dataset.metadata['uci_id']} ({dataset.metadata['name']}) due to no target variables.")
            continue

        # Get the list of target column names to iterate over
        targets_to_process_names = current_dataset_targets_df.columns.tolist()

        # model assessment loop to account for multiple target var
        for target_col_name in targets_to_process_names:
            X = dataset.data.features
            y = current_dataset_targets_df[target_col_name]

            # extract metadata dataset features
            new_row_df = pd.DataFrame([extract_features(dataset)])

            # Add features to final dataset obj
            algo_selection = pd.concat([algo_selection, new_row_df], ignore_index=True)

            if test_log:
                # define model
                name = "Logistic Regression"
                model = LogisticRegression()

                # run simulation
                # this handles preprocessing and data splitting
                model_scores["Logistic"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1)

            if test_nn:
                # define model
                name = "Neural Network"
                model = MLPClassifier()

                # run simulation
                # this handles preprocessing and data splitting
                try:
                    model_scores["Neural Network"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=1)
                except Exception as e:
                    print(f"Exception: {e}")

            if test_nn2 and use_dataset_loader:
                # define model
                name = "Neural_Network_2"
                # keras.backend.clear_session()

                model = csce_ml_project_models.NNModel().get_model()

                # run simulation
                # this handles preprocessing and data splitting
                try:
                    model_scores[name] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=0)
                except Exception as e:
                    print(f"Exception: {e}")

            if test_tree:
                # define model
                name = "Tree"
                model = DecisionTreeClassifier(criterion = "entropy", random_state=0)

                # run simulation
                # this handles preprocessing and data splitting
                model_scores["Decision Tree"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=1)


            # Determine Recommended Algorithm
            best_model, confidence = get_best_model(model_scores)
            algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Recommended Algorithm"] = best_model
            algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Expected Confidence"] = confidence
            print(f"Recommended Algorithm for {dataset.metadata["uci_id"]}: {best_model} with confidence {confidence}")



class RunLogger:


    def __init__(self):
        self.save_dir = pathlib.Path('project_run_logs')
        self.save_name = pathlib.Path('runs.log')
        self.save_path = self.save_dir/self.save_name
        self.save_dir.mkdir(parents=True, exist_ok=True)


    def append(self, data:dict):
        """Save one log entry."""
        if 'timestamp' not in data:
            data['timestamp'] = pd.Timestamp.now().isoformat()

        with open(self.save_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(data) + '\n')


    def load(self) -> pd.DataFrame:
        """Load log into a dataframe."""
        if not self.save_path.exists():
            return pd.DataFrame()
        return pd.read_json(self.save_path, lines=True)



class Runner:


    def __init__(self, config:RunConfig):
        if isinstance(config, RunConfig):
            self.config = config
            self.runlogger = RunLogger()


    def run(self, config:Optional[RunConfig]=None):
        print(f"Running")
        if config is None:
            config = self.config
        elif isinstance(config, RunConfig):
            self.config = config
        else:
            raise Exception(f"Config is {type(config) = } but should be RunConfig.")

        use_dataset_loader = True

        ids = get_id_list(method=self.config.method)
        if config.first_id is not None:
            if config.first_id in ids:
                ids = ids[ids.index(config.first_id):]

        datasets = get_datasets(ids=ids, use_dataset_loader=True, max_datasets=self.config.max_datasets)

        # exclude convergence warnings for clearer logging
        if config.filterwarnings:
            warnings.filterwarnings("ignore", category=ConvergenceWarning)

        test_log  = 0
        test_nn   = 0
        test_nn2  = 0
        test_tree = 0

        # Create final dataset obj
        algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": [], "Expected Confidence": []})
        algo_selection["Recommended Algorithm"] = algo_selection["Recommended Algorithm"].astype("string")

        # test_log = 1
        # test_nn = 1
        test_nn2  = 1
        test_tree = 1

        for dataset in datasets:
            print(f"Running: {dataset.metadata["uci_id"]}")

            model_scores = {}

            # Get the targets for the current dataset
            current_dataset_targets_df = dataset.data.targets

            # This should be done by exclusion previously, but keep just in case
            # Skip if no target variables or if the target DataFrame is empty
            if current_dataset_targets_df is None or current_dataset_targets_df.empty:
                print(f"Skipping dataset {dataset.metadata['uci_id']} ({dataset.metadata['name']}) due to no target variables.")
                continue

            # Get the list of target column names to iterate over
            targets_to_process_names = current_dataset_targets_df.columns.tolist()

            # model assessment loop to account for multiple target var
            for target_col_name in targets_to_process_names:
                X = dataset.data.features
                y = current_dataset_targets_df[target_col_name]

                # extract metadata dataset features
                new_row_df = pd.DataFrame([extract_features(dataset)])

                # Add features to final dataset obj
                algo_selection = pd.concat([algo_selection, new_row_df], ignore_index=True)

                if test_log:
                    # define model
                    name = "Logistic Regression"
                    model = LogisticRegression()

                    # run simulation
                    # this handles preprocessing and data splitting
                    model_scores["Logistic"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=0)

                if test_nn:
                    # define model
                    name = "Neural Network"
                    model = MLPClassifier()

                    # run simulation
                    # this handles preprocessing and data splitting
                    try:
                        model_scores["Neural Network"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=0)
                    except Exception as e:
                        print(f"Exception: {e}")

                if test_nn2 and use_dataset_loader:
                    # define model
                    name = "Neural_Network_2"
                    model = csce_ml_project_models.NNModel().get_model()
                    version=''
                    timestamp=pd.Timestamp.now().isoformat()

                    # run simulation
                    # this handles preprocessing and data splitting
                    id = dataset.metadata["uci_id"]
                    try:
                        model_score = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=0)
                        model_scores[name] = model_score
                        self.runlogger.append({'timestamp':timestamp, 'name':name, 'version':version, 'dataset_id':id, 'result':model_score, 'errors':None})
                    except Exception as e:
                        self.runlogger.append({'timestamp':timestamp, 'name':name, 'version':version, 'dataset_id':id, 'result':f'Exception', 'errors':f"{e}"})
                        print(f"Exception: {e}")
                        tf.keras.backend.clear_session()
                        continue
                    
                if test_tree:
                    # define model
                    name = "Tree"
                    model = DecisionTreeClassifier(criterion = "entropy", random_state=0)

                    # run simulation
                    # this handles preprocessing and data splitting
                    model_scores["Decision Tree"] = run_simulation(X, y, model, name, dataset=dataset, log_conf=1, log_all=0)

                # Determine Recommended Algorithm
                best_model, confidence = get_best_model(model_scores)
                algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Recommended Algorithm"] = best_model
                algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Expected Confidence"] = confidence
                print(f"Recommended Algorithm for {dataset.metadata["uci_id"]}: {best_model} with confidence {confidence}")



In [3]:
config1 = RunConfig(
    max_datasets=None,
    method='existing_list',
)
runner = Runner(config=config1)

In [4]:
runner.run()

Output()

Running
Using local datasets.
Save dir is: project_datasets


Exception loading: [38] [Errno 2] No such file or directory: 'project_datasets/38.pkl'

Loaded datasets: 172

Failed to load: 1: ['38']

Running: 53


I0000 00:00:1775009149.771500  118647 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5555 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1775009150.961614  118835 service.cc:152] XLA service 0x77d3ec017160 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775009150.961637  118835 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti, Compute Capability 8.6
I0000 00:00:1775009151.057294  118835 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775009152.415268  118835 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Neural_Network_2 53 confidence score (mean F1 - std F1): 0.5864
Tree 53 mean F1: 0.9321
Tree 53 std F1: 0.0439
Tree 53 confidence score (mean F1 - std F1): 0.8882
Recommended Algorithm for 53: Decision Tree with confidence 0.8881828305197754
Running: 45


KeyboardInterrupt: 